**Colab note.** After a Colab runtime reset, install segmentation-models-pytorch with `--no-deps` so Colab's CUDA build of torch is not replaced: `!pip -q install --no-deps segmentation-models-pytorch` followed by `!pip -q install timm safetensors huggingface_hub` (see the recovery cell below). Do not run a plain `pip install segmentation-models-pytorch` after a reset.

In [ ]:
# ---- Cell 1: setup ----
!nvidia-smi
!pip -q install segmentation-models-pytorch
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/yhlleo/DeepCrack.git
!ls DeepCrack/dataset

!unzip -q DeepCrack/dataset/DeepCrack.zip -d DeepCrack/dataset/
import os
for dp, dn, fn in os.walk("DeepCrack/dataset"):
    print(dp, "| subdirs:", dn[:8], "| files:", len(fn), fn[:3])

In [ ]:
# ---- Cell 2: data and split ----
import glob, os, random, numpy as np, torch
from PIL import Image
import segmentation_models_pytorch as smp

ROOT = "DeepCrack/dataset"
random.seed(0); np.random.seed(0); torch.manual_seed(0)

def pairs(split):
    labs = {os.path.splitext(os.path.basename(p))[0]: p
            for p in glob.glob(f"{ROOT}/{split}_lab/*")}
    return [(i, labs[os.path.splitext(os.path.basename(i))[0]])
            for i in sorted(glob.glob(f"{ROOT}/{split}_img/*"))]

train_all, test = pairs("train"), pairs("test")
random.shuffle(train_all)
n_val = int(0.2 * len(train_all))
val, train = train_all[:n_val], train_all[n_val:]
print(len(train), len(val), len(test))

MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
class DS(torch.utils.data.Dataset):
    def __init__(self, items, aug=False): self.items, self.aug = items, aug
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ip, lp = self.items[i]
        img = np.asarray(Image.open(ip).convert("RGB").resize((544, 384)), dtype=np.float32) / 255
        m = (np.asarray(Image.open(lp).convert("L").resize((544, 384), Image.NEAREST)) > 127).astype(np.float32)
        if self.aug:
            if random.random() < .5: img, m = img[:, ::-1], m[:, ::-1]
            if random.random() < .5: img, m = img[::-1], m[::-1]
        img = ((img - MEAN) / STD).transpose(2, 0, 1)
        return torch.tensor(img.copy(), dtype=torch.float32), torch.tensor(m.copy())[None]

import numpy as np
from PIL import Image
ip, lp = train[0]
m = np.asarray(Image.open(lp).convert("L"))
print(ip, lp, m.shape, np.unique(m), "crack fraction:", (m > 127).mean())

In [ ]:
# ---- Cell 3: model, training, test ----
dev = "cuda"
mk = lambda items, aug, sh: torch.utils.data.DataLoader(DS(items, aug), batch_size=8, shuffle=sh, num_workers=2)
tl, vl, te = mk(train, True, True), mk(val, False, False), mk(test, False, False)

model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(dev)
opt = torch.optim.AdamW(model.parameters(), 3e-4)
bce, dice_loss = torch.nn.BCEWithLogitsLoss(), smp.losses.DiceLoss("binary")

@torch.no_grad()
def evaluate(loader):
    model.eval(); ious, dices = [], []
    for x, y in loader:
        p = (torch.sigmoid(model(x.to(dev))) > 0.5).float().cpu()
        for a, b in zip(p, y):
            i = (a * b).sum().item(); u = ((a + b) > 0).sum().item(); s = a.sum().item() + b.sum().item()
            ious.append(i / (u + 1e-8)); dices.append(2 * i / (s + 1e-8))
    return float(np.mean(ious)), float(np.mean(dices))

CK = "/content/drive/MyDrive/ntnu/day1_best.pt"
os.makedirs(os.path.dirname(CK), exist_ok=True)
best = 0
for ep in range(30):
    model.train(); tot = 0
    for x, y in tl:
        x, y = x.to(dev), y.to(dev)
        out = model(x); loss = bce(out, y) + dice_loss(out, y)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    vi, vd = evaluate(vl)
    print(ep, round(tot / len(tl), 4), "val IoU", round(vi, 3), "Dice", round(vd, 3))
    if vi > best: best = vi; torch.save(model.state_dict(), CK)

model.load_state_dict(torch.load(CK))
print("TEST IoU, Dice:", evaluate(te))

In [ ]:
# ---- Cell 4: figure ----
import matplotlib.pyplot as plt
os.makedirs("results/figures", exist_ok=True)
model.eval(); x, y = next(iter(te))
with torch.no_grad(): p = (torch.sigmoid(model(x.to(dev))) > .5).cpu()
fig, ax = plt.subplots(3, 4, figsize=(12, 7))
for k in range(4):
    ax[0, k].imshow((x[k].permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)); ax[1, k].imshow(y[k, 0]); ax[2, k].imshow(p[k, 0])
    for r in range(3): ax[r, k].axis("off")
plt.savefig("results/figures/day1_deepcrack_preds.png", dpi=150, bbox_inches="tight")

In [ ]:
import json, os
os.makedirs("results/metrics", exist_ok=True)
json.dump({"model": "Unet-resnet34", "data": "DeepCrack", "split": "240/60/237 (train/val/test)",
           "seed": 0, "epochs": 30, "best_val_iou": 0.699, "test_iou": 0.675, "test_dice": 0.784},
          open("results/metrics/day1_deepcrack.json", "w"), indent=2)

In [ ]:
# ---- Recovery cell: run after any runtime reset ----
!pip -q install --no-deps segmentation-models-pytorch
!pip -q install timm safetensors huggingface_hub
import torch; print(torch.__version__, "CUDA:", torch.cuda.is_available())   # must print True
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists("DeepCrack"): os.system("git clone -q https://github.com/yhlleo/DeepCrack.git")
if not os.path.exists("DeepCrack/dataset/train_img"): os.system("unzip -q DeepCrack/dataset/DeepCrack.zip -d DeepCrack/dataset/")

import glob, json, random, numpy as np
from PIL import Image
ROOT, dev = "DeepCrack/dataset", "cuda"
random.seed(0); np.random.seed(0); torch.manual_seed(0)

def pairs(split):
    labs = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f"{ROOT}/{split}_lab/*")}
    return [(i, labs[os.path.splitext(os.path.basename(i))[0]]) for i in sorted(glob.glob(f"{ROOT}/{split}_img/*"))]

train_all, test = pairs("train"), pairs("test")
random.shuffle(train_all)
n_val = int(0.2 * len(train_all)); val, train = train_all[:n_val], train_all[n_val:]
print(len(train), len(val), len(test))          # expect 240 60 237

MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
class DS(torch.utils.data.Dataset):
    def __init__(self, items, aug=False): self.items, self.aug = items, aug
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ip, lp = self.items[i]
        img = np.asarray(Image.open(ip).convert("RGB").resize((544, 384)), dtype=np.float32) / 255
        m = (np.asarray(Image.open(lp).convert("L").resize((544, 384), Image.NEAREST)) > 127).astype(np.float32)
        img = ((img - MEAN) / STD).transpose(2, 0, 1)
        return torch.tensor(img.copy(), dtype=torch.float32), torch.tensor(m.copy())[None]
mk = lambda items, aug, sh: torch.utils.data.DataLoader(DS(items, aug), batch_size=8, shuffle=sh, num_workers=2)

# save the split so it can be reused and committed
os.makedirs("results/metrics", exist_ok=True)
json.dump({"train": [os.path.basename(a) for a, _ in train], "val": [os.path.basename(a) for a, _ in val],
           "test": [os.path.basename(a) for a, _ in test]}, open("results/metrics/split_seed0.json", "w"))

In [ ]:
# ---- Cell A: data with augmentation, losses, helpers ----
import os, json, random, numpy as np, pandas as pd, torch, torch.nn.functional as F
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp

OUT = "/content/drive/MyDrive/ntnu/runs"; os.makedirs(OUT, exist_ok=True)
bce, dice_loss = torch.nn.BCEWithLogitsLoss(), smp.losses.DiceLoss("binary")

class DS2(torch.utils.data.Dataset):
    def __init__(self, items, train=False): self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ip, lp = self.items[i]
        img = np.asarray(Image.open(ip).convert("RGB").resize((544, 384)), dtype=np.float32) / 255
        m = (np.asarray(Image.open(lp).convert("L").resize((544, 384), Image.NEAREST)) > 127).astype(np.float32)
        if self.train:
            x0 = random.randint(0, 544 - 384); img, m = img[:, x0:x0 + 384], m[:, x0:x0 + 384]
            if random.random() < .5: img, m = img[:, ::-1], m[:, ::-1]
            if random.random() < .5: img, m = img[::-1], m[::-1]
            img = np.clip(img * random.uniform(.8, 1.2) + random.uniform(-.1, .1), 0, 1)
        img = ((img - MEAN) / STD).transpose(2, 0, 1)
        return torch.tensor(img.copy(), dtype=torch.float32), torch.tensor(m.copy())[None]

def soft_skel(x, k=10):
    er = lambda t: torch.minimum(-F.max_pool2d(-t, (3, 1), 1, (1, 0)), -F.max_pool2d(-t, (1, 3), 1, (0, 1)))
    op = lambda t: F.max_pool2d(er(t), 3, 1, 1)
    s = F.relu(x - op(x))
    for _ in range(k):
        x = er(x); d = F.relu(x - op(x)); s = s + F.relu(d - s * d)
    return s

def cldice(logits, y, k=10, eps=1e-6):
    p = torch.sigmoid(logits); sp, st = soft_skel(p, k), soft_skel(y, k)
    tp = ((sp * y).sum() + eps) / (sp.sum() + eps); ts = ((st * p).sum() + eps) / (st.sum() + eps)
    return 1 - 2 * tp * ts / (tp + ts)

@torch.no_grad()
def evaluate2(model, loader):
    model.eval(); ious, dices = [], []
    for x, y in loader:
        p = (torch.sigmoid(model(x.to(dev))) > 0.5).float().cpu()
        for a, b in zip(p, y):
            i = (a * b).sum().item(); u = ((a + b) > 0).sum().item(); s = a.sum().item() + b.sum().item()
            ious.append(i / (u + 1e-8)); dices.append(2 * i / (s + 1e-8))
    return float(np.mean(ious)), float(np.mean(dices))

In [ ]:
# ---- Cell B: run function ----
ARCH = {"Unet": smp.Unet, "UnetPlusPlus": smp.UnetPlusPlus, "DeepLabV3Plus": smp.DeepLabV3Plus, "FPN": smp.FPN}

def run(arch, enc, loss_name, seed, epochs=40, bs=8, lr=3e-4):
    tag = f"{arch}_{enc}_{loss_name}_s{seed}"
    if os.path.exists(f"{OUT}/{tag}.json"): return json.load(open(f"{OUT}/{tag}.json"))
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tl = DataLoader(DS2(train, True), batch_size=bs, shuffle=True, num_workers=2, drop_last=True)
    vl, te = mk(val, False, False), mk(test, False, False)
    model = ARCH[arch](enc, encoder_weights="imagenet", in_channels=3, classes=1).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(tl))
    scaler = torch.amp.GradScaler("cuda"); best, ck = -1, f"{OUT}/{tag}.pt"
    for ep in range(epochs):
        model.train()
        for x, y in tl:
            x, y = x.to(dev), y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16): out = model(x)
            out = out.float(); loss = bce(out, y) + dice_loss(out, y)
            if "cldice" in loss_name: loss = loss + 0.5 * cldice(out, y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        vi, vd = evaluate2(model, vl)
        if vi > best: best = vi; torch.save(model.state_dict(), ck)
    model.load_state_dict(torch.load(ck)); ti, td = evaluate2(model, te)
    r = dict(tag=tag, arch=arch, enc=enc, loss=loss_name, seed=seed, val_iou=best, test_iou=ti, test_dice=td)
    json.dump(r, open(f"{OUT}/{tag}.json", "w")); print(r); return r

In [ ]:
# ---- Cell C: experiment grid (start with seed 0 only) ----
configs = [("Unet", "resnet34", "bce_dice"), ("UnetPlusPlus", "resnet34", "bce_dice"),
           ("DeepLabV3Plus", "resnet34", "bce_dice"), ("Unet", "resnet34", "bce_dice_cldice")]
res = [run(a, e, l, s) for a, e, l in configs for s in (0,)]      # later: (0, 1, 2)
df = pd.DataFrame(res)
print(df.groupby(["arch", "enc", "loss"])[["val_iou", "test_iou", "test_dice"]].agg(["mean", "std"]).round(3))
df.to_csv("results/metrics/segmentation_baselines.csv", index=False)

In [ ]:
configs = [("Unet", "resnet34", "bce_dice"), ("UnetPlusPlus", "resnet34", "bce_dice"),
           ("Unet", "resnet34", "bce_dice_cldice")]
res = [run(a, e, l, s) for a, e, l in configs for s in (0, 1, 2)]
df = pd.DataFrame(res)
print(df.groupby(["arch", "loss"])[["val_iou", "test_iou", "test_dice"]].agg(["mean", "std"]).round(3))
df.to_csv("results/metrics/segmentation_3seeds.csv", index=False)

In [ ]:
os.system("pip -q install skan")
from scipy import ndimage as ndi
from scipy.stats import spearmanr
from skimage.morphology import skeletonize, remove_small_objects
from skan import Skeleton, summarize

def crack_indices(mask, min_size=20):
    m = remove_small_objects(mask.astype(bool), min_size)
    if m.sum() == 0: return None
    H, W = m.shape
    skel = skeletonize(m)
    w = np.clip(2 * ndi.distance_transform_edt(m)[skel] - 1, 1, None)
    df_ = summarize(Skeleton(skel)); df_.columns = [c.replace("-", "_") for c in df_.columns]
    bl, el = df_["branch_distance"].to_numpy(), df_["euclidean_distance"].to_numpy()
    nb = ndi.convolve(skel.astype(int), np.ones((3, 3), int), mode="constant") - 1
    n_junc = ndi.label(skel & (nb >= 3), structure=np.ones((3, 3)))[1]
    return dict(length_px=float(bl.sum()), mean_width_px=float(w.mean()),
                tortuosity=float(bl.sum() / max(el.sum(), 1e-6)), n_branches=int(len(df_)),
                n_junctions=int(n_junc), density=float(bl.sum() / (H * W)), area_ratio=float(m.sum() / (H * W)))

# sanity check on synthetic masks (send me this output)
line = np.zeros((200, 300), bool); line[98:103, 20:280] = True
cross = line.copy(); cross[20:180, 148:153] = True
print("line :", crack_indices(line)); print("cross:", crack_indices(cross))

In [ ]:
def preds_for(arch, loss_name, seed):
    m = ARCH[arch]("resnet34", encoder_weights=None, in_channels=3, classes=1).to(dev)
    m.load_state_dict(torch.load(f"{OUT}/{arch}_resnet34_{loss_name}_s{seed}.pt")); m.eval()
    P, G = [], []
    with torch.no_grad():
        for x, y in mk(test, False, False):
            P.append((torch.sigmoid(m(x.to(dev))) > .5).cpu().numpy()[:, 0]); G.append(y.numpy()[:, 0] > .5)
    return np.concatenate(P), np.concatenate(G)

rows = []
for arch, _, loss_name in configs:
    for s in (0, 1, 2):
        P, G = preds_for(arch, loss_name, s); pairs_ = []
        for pm, gm in zip(P, G):
            a, b = crack_indices(gm), crack_indices(pm)
            if a and b: pairs_.append((a, b))
        for k in pairs_[0][0]:
            g = np.array([a[k] for a, _ in pairs_]); p = np.array([b[k] for _, b in pairs_])
            rows.append(dict(arch=arch, loss=loss_name, seed=s, index=k, n=len(g),
                             spearman=spearmanr(g, p)[0], med_rel_err=float(np.median((p - g) / np.clip(g, 1e-9, None)))))
fid = pd.DataFrame(rows)
print(fid.groupby(["index", "arch", "loss"])[["spearman", "med_rel_err"]].agg(["mean", "std"]).round(2))
fid.to_csv("results/metrics/index_fidelity.csv", index=False)

In [ ]:
def prune_spurs(skel, min_len=10, passes=2):
    skel = skel.copy()
    for _ in range(passes):
        nb = ndi.convolve(skel.astype(int), np.ones((3, 3), int), mode="constant") - 1
        sk = Skeleton(skel); d = summarize(sk, separator="_")
        short = d[(d["branch_type"] == 1) & (d["branch_distance"] < min_len)].index
        if len(short) == 0: break
        for i in short:
            c = sk.path_coordinates(i).astype(int)
            c = c[nb[c[:, 0], c[:, 1]] < 3]          # keep the junction pixel
            skel[c[:, 0], c[:, 1]] = False
        skel = skeletonize(skel)
    return skel

def branch_stats(mask, min_len):
    m = remove_small_objects(mask.astype(bool), 20)
    if m.sum() == 0: return None
    sk = prune_spurs(skeletonize(m), min_len)
    if sk.sum() < 2: return None
    nb = ndi.convolve(sk.astype(int), np.ones((3, 3), int), mode="constant") - 1
    nj = ndi.label(sk & (nb >= 3), structure=np.ones((3, 3)))[1]
    return dict(n_branches=len(summarize(Skeleton(sk), separator="_")), n_junctions=nj)

cfgs = [("Unet", "bce_dice"), ("Unet", "bce_dice_cldice")]
_, G = preds_for("Unet", "bce_dice", 0)
gt = {L: [branch_stats(g, L) for g in G] for L in (0, 10, 20)}
rows = []
for arch, loss in cfgs:
    for s in (0, 1, 2):
        P, _ = preds_for(arch, loss, s)
        for L in (0, 10, 20):
            pr = [branch_stats(p, L) for p in P]
            ok = [(a, b) for a, b in zip(gt[L], pr) if a and b]
            for k in ("n_branches", "n_junctions"):
                g = np.array([a[k] for a, _ in ok]); p = np.array([b[k] for _, b in ok])
                rows.append(dict(loss=loss, seed=s, min_len=L, index=k, gt_median=np.median(g),
                                 spearman=spearmanr(g, p)[0],
                                 med_rel_err=float(np.median((p - g) / np.clip(g, 1, None)))))
r = pd.DataFrame(rows)
print(r.groupby(["index", "loss", "min_len"])[["gt_median", "spearman", "med_rel_err"]].mean().round(2))
r.to_csv("results/metrics/branch_pruning.csv", index=False)

In [ ]:
def preds_split(arch, loss_name, seed, items):
    m = ARCH[arch]("resnet34", encoder_weights=None, in_channels=3, classes=1).to(dev)
    m.load_state_dict(torch.load(f"{OUT}/{arch}_resnet34_{loss_name}_s{seed}.pt")); m.eval()
    P, G = [], []
    with torch.no_grad():
        for x, y in mk(items, False, False):
            P.append((torch.sigmoid(m(x.to(dev))) > .5).cpu().numpy()[:, 0]); G.append(y.numpy()[:, 0] > .5)
    return np.concatenate(P), np.concatenate(G)

def idx_pairs(P, G, keys):
    out = []
    for pm, gm in zip(P, G):
        a, b = crack_indices(gm), crack_indices(pm)
        if a and b: out.append([(a[k], b[k]) for k in keys])
    return np.array(out)                       # n x keys x (gt, pred)

KEYS = ["area_ratio", "density", "mean_width_px"]
rows = []
for arch, loss in [("Unet", "bce_dice"), ("Unet", "bce_dice_cldice")]:
    for s in (0, 1, 2):
        Pv, Gv = preds_split(arch, loss, s, val); Pt, Gt = preds_split(arch, loss, s, test)
        V, T = idx_pairs(Pv, Gv, KEYS), idx_pairs(Pt, Gt, KEYS)
        for j, k in enumerate(KEYS):
            f = np.median(V[:, j, 0] / np.clip(V[:, j, 1], 1e-9, None))      # gt / pred on val
            raw = (T[:, j, 1] - T[:, j, 0]) / np.clip(T[:, j, 0], 1e-9, None)
            cal = (f * T[:, j, 1] - T[:, j, 0]) / np.clip(T[:, j, 0], 1e-9, None)
            rows.append(dict(loss=loss, seed=s, index=k, factor=f,
                             bias_raw=np.median(raw), bias_cal=np.median(cal),
                             spread_raw=np.percentile(raw, 95) - np.percentile(raw, 5),
                             spread_cal=np.percentile(cal, 95) - np.percentile(cal, 5)))
c = pd.DataFrame(rows)
print(c.groupby(["index", "loss"])[["factor", "bias_raw", "bias_cal", "spread_raw", "spread_cal"]].mean().round(3))
c.to_csv("results/metrics/index_calibration.csv", index=False)

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch
FIG, MET = "results/figures", "results/metrics"; os.makedirs(FIG, exist_ok=True)
OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00", "grey": "#666666"}
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25,
    "grid.linewidth": .5, "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "pdf.fonttype": 42})
def save(name): plt.savefig(f"{FIG}/{name}.png"); plt.savefig(f"{FIG}/{name}.pdf"); plt.show()
def panel(ax, s): ax.text(-0.14, 1.06, s, transform=ax.transAxes, fontsize=11, fontweight="bold")
LAB = {"bce_dice": "BCE+Dice", "bce_dice_cldice": "BCE+Dice+clDice"}

# ---- Fig 1: segmentation comparison, mean +/- std over 3 seeds, dots = seeds ----
d = pd.read_csv(f"{MET}/segmentation_3seeds.csv")
dl = pd.read_csv(f"{MET}/segmentation_baselines.csv").query("arch == 'DeepLabV3Plus'")
cfg = [("Unet", "bce_dice", "U-Net"), ("Unet", "bce_dice_cldice", "U-Net + clDice"), ("UnetPlusPlus", "bce_dice", "U-Net++")]
cols = [OI["blue"], OI["green"], OI["orange"]]
fig, axs = plt.subplots(1, 2, figsize=(7, 2.9))
for ax, m, t in zip(axs, ["test_iou", "test_dice"], ["Test IoU", "Test Dice"]):
    for i, (a, l, n) in enumerate(cfg):
        v = d[(d.arch == a) & (d.loss == l)][m].values
        ax.errorbar(i, v.mean(), yerr=v.std(ddof=1), fmt="o", color=cols[i], capsize=4, ms=6, zorder=3)
        ax.scatter(i + np.linspace(-.12, .12, len(v)), v, color="k", s=8, zorder=4)
    ax.scatter([3], dl[m].values, facecolors="none", edgecolors=OI["grey"], s=45, zorder=3)
    ax.set_xticks(range(4)); ax.set_xticklabels([c[2] for c in cfg] + ["DeepLabV3+\n(1 seed)"], rotation=25, ha="right")
    ax.set_ylabel(t); ax.set_xlim(-.6, 3.6)
panel(axs[0], "(a)"); panel(axs[1], "(b)"); save("fig1_segmentation_comparison")

In [ ]:
# ---- Fig 3: effect of spur pruning on branch and junction counts ----
r = pd.read_csv(f"{MET}/branch_pruning.csv")
fig, axs = plt.subplots(1, 2, figsize=(7, 2.9), sharey=True)
for ax, k, t in zip(axs, ["n_branches", "n_junctions"], ["Branches", "Junctions"]):
    for l, c in zip(["bce_dice", "bce_dice_cldice"], [OI["blue"], OI["green"]]):
        g = r[(r["index"] == k) & (r.loss == l)].groupby("min_len")["med_rel_err"].agg(["mean", "std"])
        ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", color=c, capsize=2, label=LAB[l])
    ax.axhline(0, color="k", lw=.8); ax.set_xticks([0, 10, 20]); ax.set_xlabel("Spur pruning length (px)"); ax.set_title(t)
axs[0].set_ylabel("Median relative error (pred vs GT)"); axs[0].legend(frameon=False)
panel(axs[0], "(a)"); panel(axs[1], "(b)"); save("fig3_spur_pruning")

In [ ]:
from scipy.stats import spearmanr
from PIL import Image

# ---- Fig 5: per-image GT vs predicted indices (U-Net + clDice, seed 0, test set) ----
KEYS = ["density", "mean_width_px", "area_ratio"]
NAME2 = {"density": "Length density", "mean_width_px": "Mean width (px)", "area_ratio": "Area ratio"}
Pt, Gt = preds_split("Unet", "bce_dice_cldice", 0, test)
T = idx_pairs(Pt, Gt, KEYS)
fig, axs = plt.subplots(1, 3, figsize=(8.4, 2.9))
for j, (ax, k) in enumerate(zip(axs, KEYS)):
    g, p = T[:, j, 0], T[:, j, 1]
    ax.scatter(g, p, s=9, alpha=.6, color=OI["blue"], edgecolors="none")
    lim = [0, max(g.max(), p.max()) * 1.05]; ax.plot(lim, lim, "k--", lw=.8); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Ground truth"); ax.set_ylabel("Predicted"); ax.set_title(f"{NAME2[k]}\nrho = {spearmanr(g, p)[0]:.2f}, n = {len(g)}", fontsize=9)
    panel(ax, "(" + "abc"[j] + ")")
plt.tight_layout(); save("fig5_gt_vs_pred_indices")

In [ ]:
# ---- Fig 6: qualitative results with error maps (2 random + 2 worst test images) ----
iou = np.array([(p & g).sum() / max((p | g).sum(), 1) for p, g in zip(Pt, Gt)])
rng = np.random.RandomState(0)
sel = list(rng.choice(len(iou), 2, replace=False)) + list(np.argsort(iou)[:2])
tags = ["random", "random", "worst", "2nd worst"]
fig, axs = plt.subplots(4, 4, figsize=(8, 5.8))
for r_, (i, t) in enumerate(zip(sel, tags)):
    img = np.asarray(Image.open(test[i][0]).convert("RGB").resize((544, 384)))
    p, g = Pt[i], Gt[i]
    err = np.zeros((*p.shape, 3)); err[p & g] = [1, 1, 1]; err[p & ~g] = [.84, .37, 0]; err[~p & g] = [0, .45, .7]
    for c_, (im, ttl) in enumerate([(img, "Image"), (g, "Ground truth"), (p, "Prediction"), (err, "Error map")]):
        ax = axs[r_, c_]; ax.imshow(im, cmap="gray"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values(): s.set_visible(False)
        if r_ == 0: ax.set_title(ttl)
    axs[r_, 0].set_ylabel(f"{t}\nIoU {iou[i]:.2f}", fontsize=8)
fig.legend(handles=[Patch(color="white", ec="k", label="True positive"), Patch(color=(.84, .37, 0), label="False positive"),
                    Patch(color=(0, .45, .7), label="False negative")], loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(.5, -.03))
fig.text(.5, -.07, "Images and annotations: DeepCrack (Liu et al., Neurocomputing 2019). Model: U-Net + clDice, seed 0.", ha="center", fontsize=7)
save("fig6_qualitative_error_maps")

In [ ]:
def box(ax, x, y, w, h, text, status):
    st = {"done": dict(fc="#DCEBF7", ec=OI["blue"], ls="-"), "planned": dict(fc="white", ec=OI["grey"], ls="--")}[status]
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=st["fc"], ec=st["ec"], ls=st["ls"], lw=1.2))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8)
def arrow(ax, x0, y0, x1, y1): ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", lw=1))

fig, ax = plt.subplots(figsize=(9, 3)); ax.axis("off"); ax.set_xlim(0, 10.2); ax.set_ylim(0, 3)
row1 = [("Crack images\n(DeepCrack)", "done"), ("Segmentation\nU-Net + clDice", "done"), ("Skeleton and\nmorphology indices", "done"),
        ("Calibration and\nuncertainty", "planned"), ("Damage\nfunction", "planned")]
xs = [0.1, 2.15, 4.2, 6.25, 8.3]
for x, (t, s) in zip(xs, row1): box(ax, x, 1.8, 1.6, 1.0, t, s)
for a, b in zip(xs[:-1], xs[1:]): arrow(ax, a + 1.62, 2.3, b - .02, 2.3)
box(ax, 6.25, 0.2, 1.6, 1.0, "Point-cloud geometry\n(Open3D, Fiskerkapellet)", "planned")
box(ax, 8.3, 0.2, 1.6, 1.0, "Geospatial mapping\n(GeoPandas)", "planned")
arrow(ax, 7.05, 1.22, 8.5, 1.78); arrow(ax, 9.1, 1.22, 9.1, 1.78)
ax.text(0.1, 0.35, "Solid: implemented    Dashed: planned", fontsize=8, color=OI["grey"])
save("fig7_pipeline_overview")

In [ ]:
f = pd.read_csv(f"{MET}/index_fidelity.csv").query("arch == 'Unet'")
order = ["length_px", "mean_width_px", "area_ratio", "n_branches", "n_junctions", "tortuosity"]
NAME = {"length_px": "Length", "mean_width_px": "Mean width", "area_ratio": "Area ratio",
        "n_branches": "Branches*", "n_junctions": "Junctions*", "tortuosity": "Tortuosity"}
fig, axs = plt.subplots(1, 2, figsize=(8.2, 3.4))
for ax, col, yl in zip(axs, ["spearman", "med_rel_err"], ["Spearman rho (predicted vs GT)", "Median relative error"]):
    for j, (l, c) in enumerate(zip(["bce_dice", "bce_dice_cldice"], [OI["blue"], OI["green"]])):
        g = f[f.loss == l].groupby("index")[col].agg(["mean", "std"]).loc[order]
        ax.errorbar(np.arange(len(order)) + (j - .5) * .28, g["mean"], yerr=g["std"], fmt="o", color=c, capsize=2, label=LAB[l])
    ax.set_xticks(range(len(order))); ax.set_xticklabels([NAME[o] for o in order]); plt.setp(ax.get_xticklabels(), rotation=30, ha="right"); ax.set_ylabel(yl)
axs[0].set_ylim(0, 1); axs[0].legend(frameon=False, loc="lower left"); axs[1].axhline(0, color="k", lw=.8)
fig.supxlabel("* skeletons not pruned (see Fig 3). Mean ± std over 3 seeds, test set.", fontsize=7)
panel(axs[0], "(a)"); panel(axs[1], "(b)"); save("fig2_index_fidelity")

In [ ]:
c = pd.read_csv(f"{MET}/index_calibration.csv").query("loss == 'bce_dice_cldice'")
nm = {"area_ratio": "Area ratio", "density": "Length density", "mean_width_px": "Mean width"}
fig, axs = plt.subplots(1, 2, figsize=(8, 3.3))
for ax, (raw, cal), yl, tt in zip(axs, [("bias_raw", "bias_cal"), ("spread_raw", "spread_cal")],
        ["Median relative error", "Spread of rel. error (5th-95th pct)"], ["Bias", "Per-image spread"]):
    for j, (col, lab, cc, mk_) in enumerate([(raw, "Raw", OI["red"], "o"), (cal, "Calibrated", OI["blue"], "s")]):
        g = c.groupby("index")[col].agg(["mean", "std"]).loc[list(nm)]
        ax.errorbar(np.arange(3) + (j - .5) * .2, g["mean"], yerr=g["std"], fmt=mk_, color=cc, capsize=2, label=lab)
    ax.set_xticks(range(3)); ax.set_xticklabels(nm.values()); plt.setp(ax.get_xticklabels(), rotation=20, ha="right"); ax.set_ylabel(yl); ax.set_title(tt)
axs[0].axhline(0, color="k", lw=.8); axs[0].legend(frameon=False, loc="lower left")
fig.suptitle("A global scale factor fitted on validation does not transfer to test", fontsize=10)
fig.supxlabel("U-Net + clDice, mean ± std over 3 seeds. Factor fitted on validation, applied to test.", fontsize=7)
panel(axs[0], "(a)"); panel(axs[1], "(b)"); save("fig4_calibration_null_result")

In [ ]:
import os, shutil, glob
os.makedirs("results/metrics/runs", exist_ok=True)
for p in glob.glob("/content/drive/MyDrive/ntnu/runs/*.json"):      # small JSONs only, not .pt
    shutil.copy(p, "results/metrics/runs/")
shutil.copytree("results", "/content/drive/MyDrive/ntnu/results_backup", dirs_exist_ok=True)
!zip -qr results_bundle.zip results
from google.colab import files; files.download("results_bundle.zip")